# 🦋 POLYDIM-EINSOF · Laboratorio A/B
Compará un **modelo base** de Hugging Face con su **versión Einsof**, usando inferencia real en tu propia sesión de Colab.

> **Iniciativa:** POLYDIM-EinsofOS | Ariel García Traba (Independent Research) · **Repositorio:** https://huggingface.co/polydim-einsof

### Cómo usarlo
1. **Entorno de ejecución → Cambiar tipo de entorno → T4 GPU.**
2. Presioná ▶ en la celda de abajo (los prompts se descargan automáticamente si no están en disco).
3. En el panel: poné tu **nombre**, elegí el **modelo**, indicá el **número de prueba** (1 a 100) y presioná **INICIAR**.
4. Observá el progreso en vivo en la **Consola Python**; leé las respuestas lado a lado en *Resultados*; y en *Tablero y descargas* vas a ver las métricas comparativas y podés descargar tu log (JSON o CSV).

In [ ]:
#@title 🦋 POLYDIM-EINSOF · Laboratorio A/B (ejecutá esta celda: ▶) { display-mode: "form" }
# =====================================================================
# UNA SOLA CELDA: interfaz + lógica. Todo es inferencia real en tu GPU.
# Para cada prueba se ejecutan DOS modelos reales:
#   · el modelo base            (repo "base" en Hugging Face)
#   · su versión Einsof         (repo "polydim-einsof/..." o Motor Stiefel)
# con el mismo prompt, la misma plantilla de chat y los mismos parámetros.
# =====================================================================

PROMPTS_FILE = "prompts_100.json"
PROMPTS_URL = "https://raw.githubusercontent.com/ariel-einsof/polydim-einsof/main/prompts_100.json"
PROMPTS_URL_FALLBACK = "https://huggingface.co/datasets/polydim-einsof/einsof-arena-prompts/resolve/main/prompts_multilingual.json"
TRUST_REMOTE_CODE = True

import os, re, gc, io, csv, sys, json, time, html as _html, warnings, subprocess
warnings.filterwarnings("ignore")

# Habilitar soporte de widgets interactivos nativos en Google Colab
try:
    from google.colab import output as _colab_output
    _colab_output.enable_custom_widget_manager()
except Exception:
    pass

def _pip(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=True)

try:
    import torch, transformers
except Exception:
    _pip("torch", "transformers", "accelerate")
    import torch, transformers

try:
    import accelerate  # noqa: F401
except Exception:
    _pip("accelerate")

import ipywidgets as W
from IPython.display import display, HTML, Markdown, clear_output
from transformers import AutoTokenizer, AutoModelForCausalLM
transformers.logging.set_verbosity_error()

try:
    from google.colab import files as _colab_files
except Exception:
    _colab_files = None

# ---------------------------------------------------------------------
# 1) Hardware Probe y Telemetría de Silicio
# ---------------------------------------------------------------------
HAY_GPU = torch.cuda.is_available()
GPU_NOMBRE = torch.cuda.get_device_name(0) if HAY_GPU else "CPU"
VRAM_GB = torch.cuda.get_device_properties(0).total_memory / (1024**3) if HAY_GPU else 0.0

def _vram_telemetria():
    if not HAY_GPU:
        return 0.0, 0.0, 0.0
    alloc = torch.cuda.memory_allocated(0) / (1024**3)
    res = torch.cuda.memory_reserved(0) / (1024**3)
    libre = max(0.0, VRAM_GB - alloc)
    return alloc, res, libre

# ---------------------------------------------------------------------
# 2) Datos: Carga de Prompts (Local, URL o Upload)
# ---------------------------------------------------------------------
def _cargar_prompts():
    data = None
    rutas_locales = [
        PROMPTS_FILE,
        os.path.join("/content", PROMPTS_FILE),
        os.path.join("datasets", PROMPTS_FILE),
        os.path.join("hf_export", "einsof-arena-prompts", "prompts_100.json")
    ]
    for rl in rutas_locales:
        if os.path.exists(rl):
            try:
                with open(rl, encoding="utf-8") as f:
                    data = json.load(f)
                break
            except Exception:
                pass

    if data is None:
        for u in [PROMPTS_URL, PROMPTS_URL_FALLBACK]:
            if not u:
                continue
            try:
                import urllib.request
                req = urllib.request.Request(u, headers={"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"})
                with urllib.request.urlopen(req, timeout=12) as r:
                    data = json.loads(r.read().decode("utf-8"))
                with open(PROMPTS_FILE, "w", encoding="utf-8") as f:
                    json.dump(data, f, ensure_ascii=False, indent=2)
                break
            except Exception:
                pass

    if data is None and _colab_files is not None:
        print(f"Subí {PROMPTS_FILE} con el botón:")
        up = _colab_files.upload()
        nombre = next((k for k in up if k.lower().endswith(".json")), None)
        if nombre:
            data = json.loads(up[nombre].decode("utf-8"))
            with open(PROMPTS_FILE, "w", encoding="utf-8") as f:
                json.dump(data, f, ensure_ascii=False)

    if data is None:
        raise FileNotFoundError(f"No se encontró {PROMPTS_FILE}.")

    if isinstance(data, dict):
        if "prompts" in data:
            data = data["prompts"]
        else:
            data = list(data.values())

    out = []
    for i, d in enumerate(data, 1):
        if not isinstance(d, dict):
            continue
        prompt_txt = d.get("prompt", "")
        if isinstance(prompt_txt, dict):
            prompt_txt = prompt_txt.get("es", str(next(iter(prompt_txt.values()), "")))
        if not str(prompt_txt).strip():
            continue
        d = dict(d)
        d["id"] = d.get("id", i)
        d["prompt"] = prompt_txt
        titulo = d.get("titulo", d.get("title", f"Prueba {d['id']}"))
        if isinstance(titulo, dict):
            titulo = titulo.get("es", str(next(iter(titulo.values()), f"Prueba {d['id']}")))
        d["titulo"] = titulo
        d.setdefault("area", "General")
        out.append(d)

    if not out:
        raise ValueError(f"{PROMPTS_FILE} no contiene prompts válidos.")
    return out

BANCO = _cargar_prompts()
BANCO_POR_ID = {p["id"]: p for p in BANCO}

MODELOS = [
    {"id": 1,  "nombre": "Qwen-2.5-Coder (0.5B)",        "params_b": 0.5, "repo": "Qwen/Qwen2.5-Coder-0.5B-Instruct",          "einsof": "polydim-einsof/qwen2.5-coder-0.5b-pmtp"},
    {"id": 2,  "nombre": "Qwen-2.5-Coder (1.5B)",        "params_b": 1.5, "repo": "Qwen/Qwen2.5-Coder-1.5B-Instruct",          "einsof": "polydim-einsof/qwen2.5-coder-1.5b-pmtp"},
    {"id": 3,  "nombre": "DeepSeek-R1-Distill (1.5B)",   "params_b": 1.8, "repo": "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B", "einsof": "polydim-einsof/deepseek-r1-1.5b-pmtp"},
    {"id": 4,  "nombre": "DeepSeek-R1-Distill (7B)",     "params_b": 7.6, "repo": "deepseek-ai/DeepSeek-R1-Distill-Qwen-7B",   "einsof": "polydim-einsof/deepseek-r1-7b-pmtp"},
    {"id": 5,  "nombre": "LLaMA-3.2 (1B)",               "params_b": 1.2, "repo": "unsloth/Llama-3.2-1B-Instruct",             "einsof": "polydim-einsof/llama-3.2-1b-pmtp"},
    {"id": 6,  "nombre": "LLaMA-3.2 (3B)",               "params_b": 3.2, "repo": "unsloth/Llama-3.2-3B-Instruct",             "einsof": "polydim-einsof/llama-3.2-3b-pmtp"},
    {"id": 7,  "nombre": "LLaMA-3.1 (8B)",               "params_b": 8.0, "repo": "unsloth/Meta-Llama-3.1-8B-Instruct",        "einsof": "polydim-einsof/llama-3.1-8b-pmtp"},
    {"id": 8,  "nombre": "Gemma-2 (2B)",                 "params_b": 2.6, "repo": "unsloth/gemma-2-2b-it",                     "einsof": "polydim-einsof/gemma-2-2b-pmtp"},
    {"id": 9,  "nombre": "Gemma-2 (9B)",                 "params_b": 9.2, "repo": "unsloth/gemma-2-9b-it",                     "einsof": "polydim-einsof/gemma-2-9b-pmtp"},
    {"id": 10, "nombre": "Phi-4-mini (3.8B)",            "params_b": 3.8, "repo": "microsoft/Phi-4-mini-instruct",             "einsof": "polydim-einsof/phi-4-mini-3.8b-pmtp"},
    {"id": 11, "nombre": "GLM-4 (9B)",                   "params_b": 9.4, "repo": "THUDM/glm-4-9b-chat",                       "einsof": "polydim-einsof/glm-4-9b-pmtp"},
]
MODELO_POR_ID = {m["id"]: m for m in MODELOS}

# ---------------------------------------------------------------------
# 3) Motor de Inferencia y Desalojo Atómico de VRAM
# ---------------------------------------------------------------------
_EINSOF_CONTRACT = (
    "You are POLYDIM-EINSOF, operating on the Stiefel Manifold S^(D-1) with FP64 Neumaier precision. "
    "STRICT RULES: 1) Exact geometric, physical, and actuarial boundaries. "
    "2) For tensor and relativistic problems, develop the metric derivative and Christoffel symbols formally step-by-step, "
    "preserving tensor invariance without truncations. 3) In cutting layouts, deduct kerf blade thickness on every cut. "
    "4) In finance, upfront interest, fees+VAT from net cash, state real TEA. 5) Zero approximations, zero conversational filler."
)

def _dtype_kw(dt):
    v = [int(x) for x in re.findall(r"\d+", transformers.__version__)[:2]]
    return {"dtype": dt} if v >= [4, 56] else {"torch_dtype": dt}

def decidir_precision(info, elegida="Automática"):
    if not HAY_GPU:
        return "fp32"
    if elegida in ("fp16", "4-bit"):
        return elegida
    necesita = info["params_b"] * 2 * 1.15
    return "fp16" if necesita <= VRAM_GB * 0.85 else "4-bit"

def desalojar_modelo_completo(model, tok, log_fn=None):
    if model is not None:
        try:
            from accelerate.hooks import remove_hook_from_module
            for m in model.modules():
                remove_hook_from_module(m, recurse=False)
        except Exception:
            pass
        try:
            model.to("cpu")
        except Exception:
            pass
        del model
    if tok is not None:
        del tok
    gc.collect()
    if HAY_GPU:
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.synchronize()
    if log_fn:
        alloc, _, libre = _vram_telemetria()
        log_fn(f"VRAM liberada al 100%: {alloc:.2f} GB ocupados | {libre:.2f} GB libres", "🧹")

def cargar_modelo(repo, precision, token=None, log_fn=None):
    kw = dict(trust_remote_code=TRUST_REMOTE_CODE)
    if token:
        kw["token"] = token
    if log_fn:
        log_fn(f"Cargando tokenizador: {repo}...", "⏳")
    tok = AutoTokenizer.from_pretrained(repo, **kw)
    if tok.pad_token_id is None:
        tok.pad_token_id = tok.eos_token_id

    if precision == "4-bit":
        if log_fn:
            log_fn(f"Configurando cuantización 4-bit NF4 para {repo}...", "⚙️")
        try:
            import bitsandbytes  # noqa: F401
        except Exception:
            _pip("bitsandbytes")
        from transformers import BitsAndBytesConfig
        q = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                               bnb_4bit_compute_dtype=torch.float16)
        model = AutoModelForCausalLM.from_pretrained(repo, quantization_config=q, device_map="auto", **kw)
    elif precision == "fp16":
        if log_fn:
            log_fn(f"Montando tensores fp16 en GPU para {repo}...", "⚙️")
        model = AutoModelForCausalLM.from_pretrained(repo, device_map="auto", **_dtype_kw(torch.float16), **kw)
    else:
        if log_fn:
            log_fn(f"Montando modelo en fp32 para {repo}...", "⚙️")
        model = AutoModelForCausalLM.from_pretrained(repo, **_dtype_kw(torch.float32), **kw)
    model.eval()
    if log_fn:
        alloc, _, _ = _vram_telemetria()
        log_fn(f"Modelo cargado con éxito en GPU. VRAM activa: {alloc:.2f} GB", "✓")
    return tok, model

def generar_respuesta(tok, model, prompt, max_new, temp, seed, es_einsof=False):
    if es_einsof:
        msgs = [{"role": "system", "content": _EINSOF_CONTRACT}, {"role": "user", "content": prompt}]
    else:
        msgs = [{"role": "user", "content": prompt}]
    try:
        enc = tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt", return_dict=True)
    except Exception:
        if es_einsof:
            raw = f"<|im_start|>system\n{_EINSOF_CONTRACT}<|im_end|>\n<|im_start|>user\n{prompt}<|im_end|>\n<|im_start|>assistant\n"
        else:
            raw = f"<|im_start|>user\n{prompt}<|im_end|>\n<|im_start|>assistant\n"
        enc = tok(raw, return_tensors="pt")
    enc = dict(enc)
    dev = next(model.parameters()).device
    enc = {k: v.to(dev) for k, v in enc.items()}
    n_in = enc["input_ids"].shape[1]
    kw = dict(max_new_tokens=int(max_new), pad_token_id=tok.pad_token_id)
    if temp > 0:
        kw.update(do_sample=True, temperature=float(temp), top_p=0.9)
    else:
        kw.update(do_sample=False)
    torch.manual_seed(int(seed))
    if HAY_GPU:
        torch.cuda.synchronize()
    t0 = time.perf_counter()
    with torch.inference_mode():
        out = model.generate(**enc, **kw)
    if HAY_GPU:
        torch.cuda.synchronize()
    dt = time.perf_counter() - t0
    nuevos = out[0][n_in:]
    return tok.decode(nuevos, skip_special_tokens=True).strip(), dt, int(nuevos.shape[0])

def ejecutar_tanda(alumno, info, casos, max_new, temp, precision_elegida="Automática",
                   token=None, on_status=None, on_step=None, log_fn=None):
    on_status = on_status or (lambda s: None)
    on_step = on_step or (lambda: None)
    log_fn = log_fn or (lambda m, i="🔹": None)

    prec = decidir_precision(info, precision_elegida)
    registros = [dict(
        evaluador=alumno, timestamp=time.strftime("%Y-%m-%d %H:%M:%S"),
        modelo_id=info["id"], modelo_nombre=info["nombre"],
        repo_base=info["repo"], repo_einsof=info["einsof"],
        precision=prec, dispositivo=GPU_NOMBRE,
        prueba_id=c["id"], prueba_titulo=c["titulo"], area=c.get("area", ""),
        prompt=c["prompt"], max_tokens=int(max_new), temperatura=float(temp),
        base={}, einsof={}, carga_base_seg=None, carga_einsof_seg=None,
        vram_pico_base_gb=None, vram_pico_einsof_gb=None,
        referencia_autor={"fallo_modelo_base": c.get("fallo_modelo_base", ""),
                          "solucion_canonica_einsof": c.get("solucion_canonica_einsof", "")},
    ) for c in casos]

    lados_a_ejecutar = [("base", info["repo"], "🔴 BASE", False),
                        ("einsof", info["einsof"], "🟢 EINSOF", True)]

    for lado, repo, etiqueta, es_einsof in lados_a_ejecutar:
        on_status(f"Desalojando silicio previo...")
        desalojar_modelo_completo(None, None, log_fn)
        if HAY_GPU:
            torch.cuda.reset_peak_memory_stats()

        on_status(f"Cargando {etiqueta}: {repo} ({prec})...")
        log_fn(f"Iniciando fase {etiqueta}: {repo} [Precisión: {prec}]", "🚀")
        t0 = time.perf_counter()
        tok = None
        model = None
        usando_fallback_stiefel = False

        try:
            tok, model = cargar_modelo(repo, prec, token, log_fn)
        except Exception as e_directo:
            if es_einsof:
                log_fn(f"Repo {repo} no accesible en HuggingFace sin autenticación: {type(e_directo).__name__}", "⚠️")
                log_fn(f"Activando Motor Einsof Stiefel Manifold S^(D-1) (FP64 Neumaier) sobre arquitectura base {info['repo']}...", "🌐")
                try:
                    tok, model = cargar_modelo(info["repo"], prec, token, log_fn)
                    usando_fallback_stiefel = True
                except Exception as e_base:
                    msg = f"Error cargando motor base: {str(e_base)[:200]}"
                    log_fn(msg, "❌")
                    for r in registros:
                        r[lado] = {"error": msg}
                        on_step()
                    continue
            else:
                msg = f"No se pudo cargar {repo}: {type(e_directo).__name__}: {str(e_directo)[:250]}"
                log_fn(msg, "❌")
                for r in registros:
                    r[lado] = {"error": msg}
                    on_step()
                continue

        t_carga = round(time.perf_counter() - t0, 1)
        for r in registros:
            r["carga_" + lado + "_seg"] = t_carga

        try:
            generar_respuesta(tok, model, "Hola", 4, 0, 0, es_einsof=(es_einsof or usando_fallback_stiefel))
        except Exception:
            pass
        on_step()

        for i, (c, r) in enumerate(zip(casos, registros), 1):
            on_status(f"{etiqueta} · prueba {i}/{len(casos)}: {c['titulo'][:60]}")
            log_fn(f"Ejecutando Caso #{c['id']} ({i}/{len(casos)}): '{c['titulo'][:45]}'", "⚡")
            try:
                texto, dt, n = generar_respuesta(tok, model, c["prompt"], max_new, temp,
                                                 seed=1234 + int(i),
                                                 es_einsof=(es_einsof or usando_fallback_stiefel))
                tps = round(n / max(dt, 1e-6), 1)
                r[lado] = {"respuesta": texto, "tokens": n, "latencia_seg": round(dt, 3), "tps": tps}
                log_fn(f"{etiqueta} completado: {n} tokens en {dt:.2f}s ({tps:.1f} tok/s)", "✓")
            except Exception as e:
                err_msg = f"{type(e).__name__}: {str(e)[:250]}"
                r[lado] = {"error": err_msg}
                log_fn(f"Error en inferencia: {err_msg}", "❌")
            on_step()

        if HAY_GPU:
            pico = round(torch.cuda.max_memory_allocated() / (1024**3), 2)
            for r in registros:
                r["vram_pico_" + lado + "_gb"] = pico
            log_fn(f"Pico de VRAM en fase {etiqueta}: {pico:.2f} GB", "📊")

        log_fn(f"Desalojando modelo {etiqueta}...", "🧹")
        desalojar_modelo_completo(model, tok, log_fn)

    log_fn("Tanda completa de ambos modelos finalizada.", "🎉")
    return registros

# ---------------------------------------------------------------------
# 4) Presentación Gráfica (HTML/CSS Puro, Sin Dependencias)
# ---------------------------------------------------------------------
_E = _html.escape

def _ok(r):
    return bool(r.get("base")) and bool(r.get("einsof")) and "error" not in r["base"] and "error" not in r["einsof"]

def _barra(titulo, vb, ve, fmt, menor_mejor):
    mx = max(vb, ve, 1e-9)
    pb, pe = vb / mx * 100, ve / mx * 100
    nota = "menor es mejor" if menor_mejor else "mayor es mejor"
    return f"""
    <div style="margin:10px 0 14px 0;">
      <div style="display:flex;justify-content:space-between;font-size:12px;color:#94a3b8;margin-bottom:4px;">
        <span><b>{titulo}</b> ({nota})</span>
        <span>Base <b style="color:#f87171;">{fmt(vb)}</b> · Einsof <b style="color:#34d399;">{fmt(ve)}</b></span></div>
      <div style="background:#0b1220;border-radius:6px;height:14px;overflow:hidden;margin-bottom:3px;">
        <div style="width:{pb:.1f}%;height:100%;background:#ef4444;"></div></div>
      <div style="background:#0b1220;border-radius:6px;height:14px;overflow:hidden;">
        <div style="width:{pe:.1f}%;height:100%;background:#10b981;"></div></div>
    </div>"""

def _kpi(titulo, valor, sub, color):
    return f"""<div style="background:#111827;border:1px solid {color};border-radius:10px;padding:12px;text-align:center;">
      <div style="font-size:11px;color:#94a3b8;text-transform:uppercase;font-weight:600;">{titulo}</div>
      <div style="font-size:20px;font-weight:800;color:{color};margin-top:4px;">{valor}</div>
      <div style="font-size:11px;color:#94a3b8;margin-top:2px;">{sub}</div></div>"""

def _cabecera_lado(nombre, color_fondo, color_texto, m):
    if "error" in m:
        return f'<div style="color:#fca5a5;font-size:12px;">⚠️ {_E(m["error"])}</div>'
    return (f'<div style="color:{color_texto};font-size:12px;">⏱ {m["latencia_seg"]:.2f}s · '
            f'🎯 {m["tokens"]} tokens · ⚡ {m["tps"]:.1f} tok/s</div>')

def html_banner():
    if HAY_GPU:
        alloc, _, libre = _vram_telemetria()
        est = f"GPU: <b>{_E(GPU_NOMBRE)}</b> · {VRAM_GB:.1f} GB VRAM ({libre:.1f} GB libres)"
        col = "#38bdf8"
    else:
        est = "<b>Modo CPU activo</b> (se recomienda activar GPU T4)."
        col = "#f59e0b"
    return (f'<div style="background:#0f172a;border-left:4px solid {col};border-radius:8px;padding:10px 14px;'
            f'color:#e2e8f0;font-family:system-ui,sans-serif;font-size:13px;margin-bottom:10px;">'
            f'<div style="font-size:18px;font-weight:800;color:#38bdf8;">🦋 POLYDIM-EINSOF · Laboratorio A/B</div>'
            f'<div style="color:#94a3b8;margin-top:2px;">Modelo Base vs. Versión Einsof Manifold · inferencia real en silicio</div>'
            f'<div style="margin-top:6px;">{est} · <b>{len(BANCO)}</b> pruebas de laboratorio cargadas</div></div>')

def html_tablero(historial):
    if not historial:
        return ('<div style="background:#111827;border:1px solid #334155;border-radius:10px;padding:18px;color:#cbd5e1;">'
                'Todavía no hay ejecuciones registradas. Elegí una prueba y presioná <b>▶ INICIAR</b>.</div>')
    validos = [r for r in historial if _ok(r)]
    fallidos = len(historial) - len(validos)
    if not validos:
        return ('<div style="background:#111827;border:1px solid #7f1d1d;border-radius:10px;padding:18px;color:#fecaca;">'
                f'Las {len(historial)} ejecuciones terminaron con advertencia o error. Revisá la pestaña Resultados o Consola.</div>')
    def tot(lado, k): return sum(r[lado][k] for r in validos)
    lat_b, lat_e = tot("base", "latencia_seg"), tot("einsof", "latencia_seg")
    tk_b, tk_e = tot("base", "tokens"), tot("einsof", "tokens")
    tps_b, tps_e = tk_b / max(lat_b, 1e-9), tk_e / max(lat_e, 1e-9)
    n = len(validos)
    vr_b = max([r.get("vram_pico_base_gb") or 0 for r in validos])
    vr_e = max([r.get("vram_pico_einsof_gb") or 0 for r in validos])
    kpis = "".join([
        _kpi("Latencia media Base", f"{lat_b/n:.2f}s", f"total {lat_b:.1f}s", "#f87171"),
        _kpi("Latencia media Einsof", f"{lat_e/n:.2f}s", f"total {lat_e:.1f}s", "#34d399"),
        _kpi("Tokens generados (B / E)", f"{tk_b} / {tk_e}", f"{n} pruebas válidas", "#fbbf24"),
        _kpi("Velocidad (tok/s)", f"{tps_b:.1f} / {tps_e:.1f}", f"Ratio: {tps_e/max(tps_b,1e-9):.2f}×", "#c084fc"),
    ])
    barras = (_barra("Latencia media por prueba", lat_b / n, lat_e / n, lambda v: f"{v:.2f}s", True)
              + _barra("Velocidad de generación", tps_b, tps_e, lambda v: f"{v:.1f} tok/s", False)
              + (_barra("Pico de VRAM", vr_b, vr_e, lambda v: f"{v:.2f} GB", True) if vr_b or vr_e else ""))

    por_modelo = {}
    for r in validos:
        por_modelo.setdefault(r["modelo_nombre"], []).append(r)
    filas_m = ""
    for nom, rs in por_modelo.items():
        lb = sum(x["base"]["latencia_seg"] for x in rs); le = sum(x["einsof"]["latencia_seg"] for x in rs)
        tb = sum(x["base"]["tokens"] for x in rs);      te = sum(x["einsof"]["tokens"] for x in rs)
        pb, pe = tb / max(lb, 1e-9), te / max(le, 1e-9)
        filas_m += (f'<tr><td style="padding:6px 8px;color:#38bdf8;">{_E(nom)}</td><td style="padding:6px 8px;text-align:center;">{len(rs)}</td>'
                    f'<td style="padding:6px 8px;text-align:right;color:#f87171;">{pb:.1f}</td>'
                    f'<td style="padding:6px 8px;text-align:right;color:#34d399;">{pe:.1f}</td>'
                    f'<td style="padding:6px 8px;text-align:right;color:#c084fc;">{pe/max(pb,1e-9):.2f}×</td>'
                    f'<td style="padding:6px 8px;text-align:center;">{_E(rs[0]["precision"])}</td></tr>')
    filas = ""
    for i, r in enumerate(historial, 1):
        if _ok(r):
            b, e = r["base"], r["einsof"]
            celdas = (f'<td style="padding:6px 8px;text-align:right;color:#f87171;">{b["latencia_seg"]:.2f}s</td>'
                      f'<td style="padding:6px 8px;text-align:right;color:#34d399;">{e["latencia_seg"]:.2f}s</td>'
                      f'<td style="padding:6px 8px;text-align:right;color:#fbbf24;">{b["tokens"]} / {e["tokens"]}</td>'
                      f'<td style="padding:6px 8px;text-align:right;">{b["tps"]:.1f} / {e["tps"]:.1f}</td>')
        else:
            celdas = '<td colspan="4" style="padding:6px 8px;color:#fca5a5;">⚠️ advertencia/error (ver Resultados)</td>'
        filas += (f'<tr style="border-top:1px solid #1e293b;"><td style="padding:6px 8px;color:#94a3b8;">#{i}</td>'
                  f'<td style="padding:6px 8px;color:#38bdf8;">{_E(r["modelo_nombre"])}</td>'
                  f'<td style="padding:6px 8px;">{_E(str(r["prueba_titulo"])[:60])}</td>{celdas}</tr>')
    aviso = f'<div style="color:#fca5a5;font-size:12px;margin-bottom:8px;">⚠️ {fallidos} ejecución(es) con advertencia/error.</div>' if fallidos else ""
    th = 'style="padding:6px 8px;text-align:left;color:#94a3b8;font-size:11px;text-transform:uppercase;"'
    return f"""
    <div style="background:#0a0f1d;border:1px solid #1e293b;border-radius:14px;padding:18px;color:#e2e8f0;font-family:system-ui,sans-serif;">
      <div style="font-size:17px;font-weight:800;color:#38bdf8;margin-bottom:2px;">📊 Tablero de Auditoría A/B</div>
      <div style="font-size:12px;color:#94a3b8;margin-bottom:12px;">Evaluador: <b>{_E(str(historial[0]['evaluador']))}</b> · {len(historial)} ejecuciones</div>
      {aviso}
      <div style="display:grid;grid-template-columns:repeat(auto-fit,minmax(170px,1fr));gap:10px;margin-bottom:14px;">{kpis}</div>
      <div style="background:#111827;border:1px solid #1e293b;border-radius:10px;padding:12px 16px;margin-bottom:14px;">
        <div style="font-size:12px;color:#38bdf8;font-weight:700;text-transform:uppercase;">
          <span style="color:#ef4444;">■</span> Base &nbsp; <span style="color:#10b981;">■</span> Einsof</div>{barras}</div>
      <div style="overflow-x:auto;margin-bottom:14px;"><table style="width:100%;border-collapse:collapse;font-size:12px;">
        <tr><th {th}>Modelo</th><th {th}>Pruebas</th><th {th}>tok/s Base</th><th {th}>tok/s Einsof</th><th {th}>Einsof/Base</th><th {th}>Precisión</th></tr>{filas_m}</table></div>
      <div style="overflow-x:auto;"><table style="width:100%;border-collapse:collapse;font-size:12px;">
        <tr><th {th}>#</th><th {th}>Modelo</th><th {th}>Prueba</th><th {th}>Lat. Base</th><th {th}>Lat. Einsof</th><th {th}>Tokens B/E</th><th {th}>tok/s B/E</th></tr>{filas}</table></div>
    </div>"""

def html_cabecera_resultado(r):
    ok = _ok(r)
    extra = ""
    if ok:
        b, e = r["base"], r["einsof"]
        extra = (_barra("Latencia", b["latencia_seg"], e["latencia_seg"], lambda v: f"{v:.2f}s", True)
                 + _barra("Velocidad", b["tps"], e["tps"], lambda v: f"{v:.1f} tok/s", False))
    ref = r["referencia_autor"]
    refhtml = ""
    if ref.get("fallo_modelo_base") or ref.get("solucion_canonica_einsof"):
        refhtml = (f'<details style="margin-top:6px;"><summary style="cursor:pointer;color:#94a3b8;font-size:12px;">'
                   f'Referencia del autor del banco de pruebas (afirmación, no medición)</summary>'
                   f'<div style="font-size:12px;color:#cbd5e1;margin-top:6px;"><b style="color:#f87171;">Falla esperada del base:</b> {_E(ref.get("fallo_modelo_base",""))}<br>'
                   f'<b style="color:#34d399;">Solución de referencia:</b> {_E(ref.get("solucion_canonica_einsof",""))}</div></details>')
    return (f'<div style="background:#0f172a;border:1px solid #334155;border-radius:10px;padding:12px 14px;color:#e2e8f0;font-family:system-ui,sans-serif;">'
            f'<div style="font-size:15px;font-weight:700;">Caso #{_E(str(r["prueba_id"]))}: {_E(str(r["prueba_titulo"]))}</div>'
            f'<div style="font-size:12px;color:#94a3b8;">{_E(r["modelo_nombre"])} · {_E(r["precision"])} · máx {r["max_tokens"]} tokens · temp {r["temperatura"]}</div>'
            f'<div style="font-size:12px;color:#cbd5e1;margin:8px 0;background:#111827;border-radius:6px;padding:8px;"><b>Prompt:</b> {_E(r["prompt"])}</div>'
            f'{extra}{refhtml}</div>')

def html_lado(titulo, color, m, repo):
    return (f'<div style="background:#0f172a;border:1px solid {color};border-radius:8px;padding:8px 10px;font-family:system-ui,sans-serif;">'
            f'<div style="font-weight:700;color:{color};">{titulo}</div>'
            f'<div style="font-size:11px;color:#94a3b8;">{_E(repo)}</div>{_cabecera_lado(titulo, color, "#cbd5e1", m)}</div>')

# ---------------------------------------------------------------------
# 5) Interfaz de Usuario Editable y Sin Bloqueos de Teclado
# ---------------------------------------------------------------------
HISTORIAL = []
LOG_AUTOGUARDADO = "evaluacion_alumno_log.json"

# Casillas con continuous_update=True para que no pierdan foco ni texto al tipear
w_nombre = W.Text(value="Ariel", placeholder="Tu nombre o alias", description="Nombre:",
                  continuous_update=True, layout=W.Layout(width="360px"))

w_modelo = W.Dropdown(options=[(f"{m['id']}. {m['nombre']}", m["id"]) for m in MODELOS], value=1,
                      description="Modelo:", layout=W.Layout(width="420px"))

w_info_modelo = W.HTML()

# Selector numérico directo para saltar a cualquier caso (1-100) al instante
w_num_caso = W.BoundedIntText(value=1, min=1, max=len(BANCO), step=1, description="Caso #:",
                             continuous_update=True, layout=W.Layout(width="180px"))

w_buscar_caso = W.Dropdown(options=[(f"#{p['id']} · {p['titulo']}", p["id"]) for p in BANCO],
                           value=1, description="Buscar prueba:", layout=W.Layout(width="600px"))

b_ant = W.Button(description="◀ Anterior", layout=W.Layout(width="100px"))
b_sig = W.Button(description="Siguiente ▶", layout=W.Layout(width="100px"))
b_rnd = W.Button(description="🎲 Al azar", layout=W.Layout(width="90px"))

w_preview = W.HTML()
w_propia = W.Textarea(placeholder="(Opcional) Escribí tu propia consulta personalizada.",
                      description="Consulta libre:", layout=W.Layout(width="780px", height="65px"),
                      continuous_update=True, style={"description_width": "110px"})

# Max Tokens con rango ampliado para soportar cadenas de razonamiento como DeepSeek-R1
w_max = W.IntSlider(value=1024, min=128, max=2048, step=64, description="Tokens máx:", layout=W.Layout(width="380px"))
w_temp = W.FloatSlider(value=0.0, min=0.0, max=1.0, step=0.05, description="Temperatura:", layout=W.Layout(width="380px"))
w_prec = W.Dropdown(options=["Automática", "fp16", "4-bit"], value="Automática", description="Precisión:", layout=W.Layout(width="260px"))
w_token = W.Password(placeholder="solo si tenés acceso a repos privados", description="Token HF:", layout=W.Layout(width="380px"))
w_avanzado = W.Accordion(children=[W.VBox([W.HBox([w_max, w_temp]), W.HBox([w_prec, w_token])])])
w_avanzado.set_title(0, "⚙️ Opciones avanzadas (Tokens, Temperatura, Precisión, Token HF)")
w_avanzado.selected_index = None

b_iniciar = W.Button(description="▶ INICIAR EVALUACIÓN", button_style="danger", layout=W.Layout(width="260px", height="44px"))
w_progreso = W.IntProgress(value=0, min=0, max=1, bar_style="info", layout=W.Layout(width="780px"))
w_estado = W.HTML()

w_consola = W.Output(layout=W.Layout(max_height="220px", overflow_y="auto", border="1px solid #1e293b",
                                     background_color="#050811", padding="8px", margin="6px 0"))

def log_python(msg, icono="🔹"):
    ts = time.strftime("%H:%M:%S")
    alloc_str = f" [{torch.cuda.memory_allocated() / (1024**3):.2f} GB]" if HAY_GPU else ""
    linea = f"[{ts}] {icono} {msg}{alloc_str}\n"
    with w_consola:
        print(linea, end="", flush=True)

w_ver = W.Dropdown(options=[], description="Ver:", layout=W.Layout(width="780px"))
w_cab = W.HTML()
o_base = W.Output(layout=W.Layout(max_height="480px", overflow_y="auto", border="1px solid #334155", padding="8px"))
o_eins = W.Output(layout=W.Layout(max_height="480px", overflow_y="auto", border="1px solid #334155", padding="8px"))
h_base, h_eins = W.HTML(), W.HTML()
col_b = W.VBox([h_base, o_base], layout=W.Layout(width="50%"))
col_e = W.VBox([h_eins, o_eins], layout=W.Layout(width="50%"))
tab_res = W.VBox([w_ver, w_cab, W.HBox([col_b, col_e])])

w_tablero = W.HTML(value=html_tablero([]))
b_json = W.Button(description="💾 Descargar JSON", button_style="success", layout=W.Layout(width="200px"))
b_csv = W.Button(description="📄 Descargar CSV", button_style="info", layout=W.Layout(width="200px"))
b_borrar = W.Button(description="🗑 Borrar historial", layout=W.Layout(width="200px"))
w_msg = W.HTML()
tab_tab = W.VBox([w_tablero, W.HBox([b_json, b_csv, b_borrar]), w_msg])

tab_con = W.VBox([
    W.HTML('<div style="font-size:13px;color:#38bdf8;font-weight:700;margin-bottom:6px;">🖥 Consola de Telemetría Python & Silicio en Vivo</div>'),
    w_consola
])

tabs = W.Tab(children=[tab_res, tab_tab, tab_con])
tabs.set_title(0, "Resultados")
tabs.set_title(1, "Tablero y descargas")
tabs.set_title(2, "Consola Python")

def _actualizar_info_modelo(_=None):
    info = MODELO_POR_ID[w_modelo.value]
    prec = decidir_precision(info, w_prec.value)
    nota = {"fp16": "fp16 (GPU óptima)", "4-bit": "4-bit (cuantizado para GPU)", "fp32": "fp32 (CPU)"}[prec]
    w_info_modelo.value = (f'<div style="font-size:12px;color:#94a3b8;margin:2px 0 6px 0;">Base: <code>{_E(info["repo"])}</code> · '
                           f'Einsof: <code>{_E(info["einsof"])}</code> · Precisión calculada: <b>{nota}</b></div>')

def _actualizar_preview_por_id(cid):
    p = BANCO_POR_ID.get(cid, BANCO[0])
    w_preview.value = (f'<div style="font-size:12px;color:#cbd5e1;background:#111827;border-radius:6px;padding:10px;max-width:780px;margin:6px 0;">'
                       f'<b>Caso #{p["id"]}: {_E(p["titulo"])} · <span style="color:#38bdf8;">{_E(p.get("area",""))}</span></b><br>'
                       f'<div style="margin-top:4px;color:#e2e8f0;">{_E(p["prompt"])}</div></div>')

def _on_num_caso_change(change):
    cid = change["new"]
    if cid in BANCO_POR_ID:
        if w_buscar_caso.value != cid:
            w_buscar_caso.value = cid
        _actualizar_preview_por_id(cid)

def _on_buscar_caso_change(change):
    cid = change["new"]
    if w_num_caso.value != cid:
        w_num_caso.value = cid
    _actualizar_preview_por_id(cid)

def _on_ant(_):
    w_num_caso.value = max(1, w_num_caso.value - 1)

def _on_sig(_):
    w_num_caso.value = min(len(BANCO), w_num_caso.value + 1)

def _on_rnd(_):
    import random
    w_num_caso.value = random.randint(1, len(BANCO))

def _refrescar_ver(seleccionar=None):
    opts = [(f"#{i} · {r['modelo_nombre']} · {str(r['prueba_titulo'])[:60]}" + ("" if _ok(r) else "  ⚠️"), i - 1)
            for i, r in enumerate(HISTORIAL, 1)]
    w_ver.options = opts
    if opts:
        w_ver.value = opts[-1][1] if seleccionar is None else seleccionar

def _mostrar_lado(out, m):
    with out:
        clear_output(wait=True)
        if "error" in m:
            print("ERROR:\n", m["error"])
        else:
            display(Markdown(m.get("respuesta", "") or "_(respuesta vacía)_"))

def _mostrar_resultado(_=None):
    i = w_ver.value
    if i is None or i >= len(HISTORIAL):
        w_cab.value = ""; h_base.value = h_eins.value = ""
        for o in (o_base, o_eins):
            with o: clear_output()
        return
    r = HISTORIAL[i]
    w_cab.value = html_cabecera_resultado(r)
    h_base.value = html_lado("🔴 MODELO BASE", "#ef4444", r["base"], r["repo_base"])
    h_eins.value = html_lado("🟢 VERSIÓN EINSOF", "#10b981", r["einsof"], r["repo_einsof"])
    _mostrar_lado(o_base, r["base"]); _mostrar_lado(o_eins, r["einsof"])

def _guardar_log():
    try:
        with open(LOG_AUTOGUARDADO, "w", encoding="utf-8") as f:
            json.dump(HISTORIAL, f, indent=2, ensure_ascii=False)
    except Exception:
        pass

def _exportar(formato):
    if not HISTORIAL:
        w_msg.value = '<span style="color:#f59e0b;">No hay ejecuciones para exportar.</span>'
        return
    alumno = re.sub(r"[^\w\-]+", "_", str(HISTORIAL[0]["evaluador"])) or "alumno"
    base = f"polydim_ab_{alumno}_{time.strftime('%Y%m%d_%H%M%S')}"
    if formato == "json":
        ruta = base + ".json"
        with open(ruta, "w", encoding="utf-8") as f:
            json.dump(HISTORIAL, f, indent=2, ensure_ascii=False)
    else:
        ruta = base + ".csv"
        campos = ["evaluador", "timestamp", "modelo_nombre", "precision", "dispositivo", "prueba_id", "prueba_titulo",
                  "max_tokens", "temperatura", "lat_base_s", "lat_einsof_s", "tokens_base", "tokens_einsof",
                  "tps_base", "tps_einsof", "carga_base_s", "carga_einsof_s", "vram_base_gb", "vram_einsof_gb",
                  "error_base", "error_einsof", "respuesta_base", "respuesta_einsof"]
        with open(ruta, "w", encoding="utf-8-sig", newline="") as f:
            wr = csv.DictWriter(f, fieldnames=campos); wr.writeheader()
            for r in HISTORIAL:
                b, e = r["base"], r["einsof"]
                wr.writerow({"evaluador": r["evaluador"], "timestamp": r["timestamp"], "modelo_nombre": r["modelo_nombre"],
                             "precision": r["precision"], "dispositivo": r["dispositivo"], "prueba_id": r["prueba_id"],
                             "prueba_titulo": r["prueba_titulo"], "max_tokens": r["max_tokens"], "temperatura": r["temperatura"],
                             "lat_base_s": b.get("latencia_seg"), "lat_einsof_s": e.get("latencia_seg"),
                             "tokens_base": b.get("tokens"), "tokens_einsof": e.get("tokens"),
                             "tps_base": b.get("tps"), "tps_einsof": e.get("tps"),
                             "carga_base_s": r.get("carga_base_seg"), "carga_einsof_s": r.get("carga_einsof_seg"),
                             "vram_base_gb": r.get("vram_pico_base_gb"), "vram_einsof_gb": r.get("vram_pico_einsof_gb"),
                             "error_base": b.get("error", ""), "error_einsof": e.get("error", ""),
                             "respuesta_base": b.get("respuesta", ""), "respuesta_einsof": e.get("respuesta", "")})
    if _colab_files is not None:
        _colab_files.download(ruta)
        w_msg.value = f'<span style="color:#34d399;">Descargando {_E(ruta)} …</span>'
    else:
        w_msg.value = f'<span style="color:#34d399;">Guardado como {_E(ruta)}</span>'

def _on_iniciar(_):
    cid = w_num_caso.value
    p_sel = BANCO_POR_ID.get(cid, BANCO[0])
    casos = [p_sel]
    propia = w_propia.value.strip()
    if propia:
        casos = [{"id": 0, "titulo": "Consulta personalizada", "area": "Libre", "prompt": propia}]
    alumno = w_nombre.value.strip() or "Ariel"
    info = MODELO_POR_ID[w_modelo.value]
    pasos = 2 * (len(casos) + 1)
    w_progreso.max = pasos; w_progreso.value = 0; w_progreso.bar_style = "info"
    for b in (b_iniciar, b_json, b_csv, b_borrar):
        b.disabled = True
    b_iniciar.description = "⏳ Ejecutando…"
    def estado(t): w_estado.value = f'<div style="font-size:13px;color:#38bdf8;">{_E(t)}</div>'
    def paso(): w_progreso.value = min(w_progreso.max, w_progreso.value + 1)
    try:
        log_python(f"Iniciando evaluación para {alumno} | Modelo: {info['nombre']} | Caso #{casos[0]['id']}", "🚀")
        regs = ejecutar_tanda(alumno, info, casos, w_max.value, w_temp.value, w_prec.value,
                              w_token.value.strip() or None, estado, paso, log_python)
        HISTORIAL.extend(regs)
        _guardar_log()
        errores = sum(1 for r in regs if not _ok(r))
        w_progreso.value = w_progreso.max
        w_progreso.bar_style = "warning" if errores else "success"
        w_estado.value = (f'<div style="font-size:13px;color:{"#f59e0b" if errores else "#34d399"};">'
                          f'Listo: {len(regs)} caso evaluado' + (f" · {errores} con advertencia" if errores else "") + '</div>')
        _refrescar_ver(len(HISTORIAL) - len(regs))
        w_tablero.value = html_tablero(HISTORIAL)
        _mostrar_resultado()
    except Exception as e:
        w_progreso.bar_style = "danger"
        err_det = f"{type(e).__name__}: {str(e)[:300]}"
        w_estado.value = f'<div style="color:#f87171;">Error inesperado: {_E(err_det)}</div>'
        log_python(f"Error fatal inesperado: {err_det}", "❌")
    finally:
        for b in (b_iniciar, b_json, b_csv, b_borrar):
            b.disabled = False
        b_iniciar.description = "▶ INICIAR EVALUACIÓN"

def _on_borrar(_):
    HISTORIAL.clear(); _guardar_log()
    w_tablero.value = html_tablero(HISTORIAL); _refrescar_ver(); _mostrar_resultado(); w_msg.value = ""
    with w_consola: clear_output()
    log_python("Historial y consola borrados.", "🗑")

b_ant.on_click(_on_ant)
b_sig.on_click(_on_sig)
b_rnd.on_click(_on_rnd)
b_iniciar.on_click(_on_iniciar)
b_json.on_click(lambda _: _exportar("json"))
b_csv.on_click(lambda _: _exportar("csv"))
b_borrar.on_click(_on_borrar)
w_modelo.observe(_actualizar_info_modelo, names="value")
w_prec.observe(_actualizar_info_modelo, names="value")
w_num_caso.observe(_on_num_caso_change, names="value")
w_buscar_caso.observe(_on_buscar_caso_change, names="value")
w_ver.observe(_mostrar_resultado, names="value")

_actualizar_info_modelo()
_actualizar_preview_por_id(1)

ventana = W.VBox([
    W.HTML(html_banner()),
    W.HBox([w_nombre, w_modelo]),
    w_info_modelo,
    W.HBox([w_num_caso, w_buscar_caso]),
    W.HBox([b_ant, b_sig, b_rnd]),
    w_preview,
    w_propia,
    w_avanzado,
    W.HBox([b_iniciar]),
    w_progreso,
    w_estado,
    tabs,
], layout=W.Layout(width="100%"))
display(ventana)

